In [ ]:
# Rasterize Pasture data and copy for each year 2000-2020

In [ ]:
import geopandas as gpd
import rasterio
from rasterio.features import rasterize
import shutil

# load shapefile
gdf = gpd.read_file("Range%3A_Unit_Pasture.shp")

with rasterio.open("USA_1km_raster.tif") as src:
    transform = src.transform
    width, height = src.width, src.height
    crs = src.crs
meta = {
    "driver": "GTiff",
    "height": height,
    "width": width,
    "count": 1,
    "dtype": "uint8",
    "crs": crs,
    "transform": transform
}
gdf = gdf.to_crs(crs)

# rasterize
shapes = ((geom, 1) for geom in gdf.geometry)
pasture_raster = rasterize(
    shapes=shapes,
    out_shape=(height, width),
    transform=transform,
    fill=0,
    dtype="uint8"
)

# save the raster
with rasterio.open("pasture_presence_2024.tif", "w", **meta) as dst:
    dst.write(pasture_raster, 1)

# copy for each year
for yr in range(2000, 2021):
    shutil.copy("pasture_presence_2024.tif", f"pasture_presence_{yr}.tif")
